# 04 – Pipeline so sánh: YOLOv8n

Hai biến thể, đều quy về **độ chính xác theo từng ô** trên cùng tập test PKLot để so với `03_train_classifier`:

| Biến thể | Cách làm |
|---|---|
| **YOLOv8n-PKLot** | Fine-tune YOLOv8n trên `data.yaml` của PKLot, phát hiện trực tiếp ô `vacant` / `occupied`. Mỗi ô thật được khớp với box dự đoán có IoU lớn nhất (≥ 0.5) |
| **YOLOv8n-COCO** | Không train. Phát hiện xe (car, motorcycle, bus, truck); ô **có xe** nếu một box xe che ≥ `COVER_THR` diện tích ô (ngưỡng chọn trên tập valid) |

Cuối notebook xuất 2 file ONNX cho `src/vision/yolo_detector.py`.

**Trên Kaggle:** *Add Input* → `pklot-yolo`; *Accelerator* → **GPU T4**; *Internet* → **On** (cài `ultralytics`, tải trọng số).
Ước tính ~1–1,5 giờ với cấu hình mặc định.

## 1. Cài đặt và cấu hình

In [ ]:
import subprocess, sys
try:
    import ultralytics
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    import ultralytics
print("ultralytics", ultralytics.__version__)

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
EPOCHS = 20
IMGSZ = 640                 # ô PUCPR rộng ~35–45 px ở 1280 → ~20 px ở 640, vẫn phát hiện được
BATCH = 32
CONF, NMS_IOU = 0.25, 0.5
MATCH_IOU = 0.5             # khớp box dự đoán với ô thật
VEHICLES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}   # lớp COCO
COVER_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
COCO_IMGSZ = [640, 1280]    # xe ở xa rất nhỏ: thử local cho thấy ở 640 YOLO COCO bỏ sót phần lớn xe
VALID_IMAGES_FOR_TUNING = 300
DEVICE = 0 if torch.cuda.is_available() else "cpu"
SPLITS = ["train", "valid", "test"]
CLASSES = ["vacant", "occupied"]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = WORK_DIR / "outputs_yolo"
OUT.mkdir(parents=True, exist_ok=True)

def find_data_root() -> Path:
    for depth in range(1, 6):
        for yaml_path in Path("/kaggle/input").glob("*/" * depth + "data.yaml"):
            if (yaml_path.parent / "labels").is_dir():
                return yaml_path.parent
    raise FileNotFoundError("Không thấy data.yaml – hãy Add Input dataset pklot-yolo (output của 00_download_pklot).")

DATA_ROOT = find_data_root()
# /kaggle/input chỉ đọc và `path: .` trong data.yaml gốc là đường dẫn tương đối → viết file yaml mới với đường dẫn tuyệt đối
DATA_YAML = WORK_DIR / "pklot.yaml"
DATA_YAML.write_text(f"path: {DATA_ROOT}\ntrain: images/train\nval: images/valid\ntest: images/test\n"
                     "names:\n  0: vacant\n  1: occupied\n")
print("DATA_ROOT =", DATA_ROOT)

## 2. Hàm khớp box và tính độ chính xác theo ô

In [ ]:
def load_gt(label_path: Path, w: int, h: int):
    # Label YOLO chuẩn hóa → box xyxy pixel + lớp
    rows = np.loadtxt(label_path, ndmin=2) if label_path.stat().st_size else np.zeros((0, 5))
    cls = rows[:, 0].astype(int)
    xc, yc, bw, bh = rows[:, 1] * w, rows[:, 2] * h, rows[:, 3] * w, rows[:, 4] * h
    return np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], 1), cls

def pairwise(a, b):
    # Trả về (diện tích giao, diện tích a, diện tích b) cho mọi cặp box xyxy
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(rb - lt, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter, area(a)[:, None], area(b)[None, :]

def slots_from_pklot_model(gt_boxes, pred_boxes, pred_cls):
    # Mỗi ô thật lấy lớp của box dự đoán có IoU lớn nhất; không khớp được (IoU < MATCH_IOU) → -1
    if len(pred_boxes) == 0:
        return np.full(len(gt_boxes), -1)
    inter, a, b = pairwise(gt_boxes, pred_boxes)
    iou = inter / (a + b - inter + 1e-9)
    best = iou.argmax(1)
    return np.where(iou.max(1) >= MATCH_IOU, pred_cls[best], -1)

def coverage_from_coco_model(gt_boxes, vehicle_boxes):
    # Tỷ lệ diện tích ô bị box xe che nhiều nhất
    if len(vehicle_boxes) == 0:
        return np.zeros(len(gt_boxes))
    inter, a, _ = pairwise(gt_boxes, vehicle_boxes)
    return (inter / (a + 1e-9)).max(1)

def slot_metrics(y, pred):
    # Ô không khớp (pred = -1) tính là dự đoán sai
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred != 1) & (y == 1)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"accuracy": float((pred == y).mean()), "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "unmatched": float((pred == -1).mean())}

PRED_CHUNK = 16   # ultralytics gộp cả danh sách ảnh thành MỘT batch → phải chia nhỏ, không thì hết bộ nhớ GPU

def run_on_split(model, split, per_image, limit=None, classes=None, imgsz=IMGSZ):
    # Chạy model trên ảnh của split, gọi per_image(gt_boxes, result) → bảng kết quả từng ô
    images = sorted((DATA_ROOT / "images" / split).glob("*.jpg"))[:limit]
    rows = []
    with tqdm(total=len(images), desc=f"{split} @{imgsz}") as bar:
        for start in range(0, len(images), PRED_CHUNK):
            chunk = [str(p) for p in images[start:start + PRED_CHUNK]]
            for res in model.predict(chunk, imgsz=imgsz, conf=CONF, iou=NMS_IOU, max_det=300, classes=classes,
                                     device=DEVICE, verbose=False):
                path = Path(res.path)
                h, w = res.orig_shape
                gt_boxes, gt_cls = load_gt(DATA_ROOT / "labels" / split / f"{path.stem}.txt", w, h)
                for i, value in enumerate(per_image(gt_boxes, res)):
                    rows.append((path.stem.split("_")[0], path.stem, int(gt_cls[i]), value))
                bar.update(1)
    return pd.DataFrame(rows, columns=["camera", "frame", "y", "value"])

RESULTS = []

def record(model_name, df, pred_col="pred"):
    for cam, g in [("all", df)] + list(df.groupby("camera")):
        RESULTS.append({"experiment": "in-domain", "model": model_name, "test_camera": cam, "n": len(g),
                        **slot_metrics(g["y"].to_numpy(), g[pred_col].to_numpy())})

## 3. YOLOv8n-PKLot: fine-tune

Nếu đã train ở lần chạy trước: *Add Input* → output của lần chạy đó (có `yolov8n_pklot/weights/best.pt`)
→ notebook dùng lại trọng số và bỏ qua bước train (~1 giờ).

In [ ]:
import gc

def find_previous_weights():
    for depth in range(1, 8):
        for p in Path("/kaggle/input").glob("*/" * depth + "best.pt"):
            if "yolov8n_pklot" in p.parts:
                return p
    return None

previous = find_previous_weights()
if previous:
    print("Dùng lại trọng số đã train:", previous)
    best_path = OUT / "yolov8n_pklot" / "weights" / "best.pt"
    best_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(previous, best_path)
else:
    model = YOLO("yolov8n.pt")
    model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, device=DEVICE,
                project=str(OUT), name="yolov8n_pklot", exist_ok=True, workers=4, plots=True)
    best_path = OUT / "yolov8n_pklot" / "weights" / "best.pt"
    del model   # giải phóng bộ nhớ GPU của lúc train trước khi đánh giá
    gc.collect()
    torch.cuda.empty_cache()
best = YOLO(str(best_path))
test_metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE, plots=False)
det = {"mAP50": float(test_metrics.box.map50), "mAP50-95": float(test_metrics.box.map)}
print("Test PKLot:", det)

In [ ]:
pk = run_on_split(best, "test", lambda gt, res: slots_from_pklot_model(gt, res.boxes.xyxy.cpu().numpy(),
                                                                      res.boxes.cls.cpu().numpy().astype(int)))
pk = pk.rename(columns={"value": "pred"})
record("yolov8n_pklot", pk)
pd.DataFrame(RESULTS).round(4)

## 4. YOLOv8n-COCO: không train, luật che phủ

Với mỗi kích thước ảnh vào, chọn `COVER_THR` trên tập **valid** (không nhìn tập test), rồi áp dụng cho test.

In [ ]:
coco = YOLO("yolov8n.pt")
coverage = lambda gt, res: coverage_from_coco_model(gt, res.boxes.xyxy.cpu().numpy())
COVER_THR = {}
for size in COCO_IMGSZ:
    val_cov = run_on_split(coco, "valid", coverage, limit=VALID_IMAGES_FOR_TUNING, classes=list(VEHICLES), imgsz=size)
    tuning = pd.DataFrame([{"cover_thr": t, **slot_metrics(val_cov["y"].to_numpy(), (val_cov["value"] >= t).astype(int).to_numpy())}
                           for t in COVER_GRID])
    COVER_THR[size] = float(tuning.loc[tuning["accuracy"].idxmax(), "cover_thr"])
    print(f"imgsz {size}: chọn COVER_THR = {COVER_THR[size]}")
    display(tuning.round(4))
    test_cov = run_on_split(coco, "test", coverage, classes=list(VEHICLES), imgsz=size)
    test_cov["pred"] = (test_cov["value"] >= COVER_THR[size]).astype(int)
    record(f"yolov8n_coco_{size}", test_cov)

## 5. Tổng hợp và xuất ONNX

In [ ]:
res = pd.DataFrame(RESULTS)
res.to_csv(OUT / "results_yolo.csv", index=False)
print("Độ chính xác theo ô trên test PKLot (so với results.csv của notebook 03):")
display(res.pivot(index="model", columns="test_camera", values="accuracy").round(4))
display(res.query("test_camera == 'all'")[["model", "accuracy", "precision", "recall", "f1", "unmatched"]].round(4))

onnx_dir = OUT / "onnx"
onnx_dir.mkdir(exist_ok=True)
exports = [("yolov8n_pklot", best, IMGSZ)] + [(f"yolov8n_coco_{s}", coco, s) for s in COCO_IMGSZ]
for name, weights, size in exports:
    path = Path(weights.export(format="onnx", imgsz=size, opset=12, dynamic=False, simplify=False))
    shutil.copy(path, onnx_dir / f"{name}.onnx")
(onnx_dir / "yolo.json").write_text(json.dumps({
    "imgsz_pklot": IMGSZ, "conf": CONF, "nms_iou": NMS_IOU, "match_iou": MATCH_IOU, "cover_thr_coco": COVER_THR,
    "vehicle_classes": VEHICLES, "test_detection_pklot": det}, indent=2))
print("Đã xuất:", sorted(p.name for p in onnx_dir.iterdir()))

## Kết quả cần tải về
Tab *Output* → `outputs_yolo/`:
- `onnx/yolov8n_pklot.onnx`, `onnx/yolov8n_coco_640.onnx`, `onnx/yolov8n_coco_1280.onnx`, `onnx/yolo.json` → `smart-parking/models/yolo/`.
  Khi chạy worker với bản COCO, đặt `--cover-thr` bằng giá trị trong `yolo.json`.
- `results_yolo.csv` → ghép với `results.csv` của notebook 03 thành bảng so sánh trong báo cáo.
- `yolov8n_pklot/` → đường cong train, confusion matrix, mAP của Ultralytics.

**Lưu ý khi viết báo cáo:** YOLOv8n-PKLot học trên đúng 3 camera của PKLot nên dễ "học thuộc" vị trí ô;
trên camera mới nên dùng biến thể COCO hoặc phân loại ô.